# Lesson 9.5 - Graph AI and GraphRAG Pipeline Demo

## Learning Objectives
- Build a toy knowledge graph from text records.
- Implement hybrid retrieval (graph neighborhood + TF-IDF).
- Generate grounded answers from retrieved context.

In [1]:
from __future__ import annotations

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1) Build a Tiny Knowledge Graph from Documents

In [2]:
docs = [
    {"id": "d1", "text": "Payment API depends on Auth Service and Redis cache."},
    {"id": "d2", "text": "Auth Service writes sessions to Redis cache."},
    {"id": "d3", "text": "Checkout Service calls Payment API for charge authorization."},
    {"id": "d4", "text": "Fraud Engine monitors Checkout Service and Payment API events."},
]

relations = [
    ("Payment API", "depends_on", "Auth Service"),
    ("Payment API", "depends_on", "Redis cache"),
    ("Auth Service", "writes_to", "Redis cache"),
    ("Checkout Service", "calls", "Payment API"),
    ("Fraud Engine", "monitors", "Checkout Service"),
    ("Fraud Engine", "monitors", "Payment API"),
]

G = nx.DiGraph()
for h, r, t in relations:
    G.add_edge(h, t, relation=r)

print({"nodes": G.number_of_nodes(), "edges": G.number_of_edges()})

{'nodes': 5, 'edges': 6}


## 2) Vector Retrieval over Raw Text Chunks

In [3]:
corpus = [d["text"] for d in docs]
ids = [d["id"] for d in docs]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)


def vector_retrieve(query: str, k: int = 2):
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, X).flatten()
    idx = np.argsort(-sims)[:k]
    return [(ids[i], corpus[i], float(sims[i])) for i in idx]

vector_retrieve("checkout payment authorization latency")

[('d3',
  'Checkout Service calls Payment API for charge authorization.',
  0.5961719285636314),
 ('d4',
  'Fraud Engine monitors Checkout Service and Payment API events.',
  0.2871192525946602)]

## 3) Graph Neighborhood Retrieval

In [4]:
entity_to_doc = {
    "Payment API": "d1",
    "Auth Service": "d2",
    "Redis cache": "d2",
    "Checkout Service": "d3",
    "Fraud Engine": "d4",
}

def graph_retrieve(seed_entity: str, hops: int = 1):
    nodes = {seed_entity}
    frontier = {seed_entity}
    for _ in range(hops):
        nxt = set()
        for n in frontier:
            nxt.update(G.successors(n))
            nxt.update(G.predecessors(n))
        nodes.update(nxt)
        frontier = nxt
    doc_ids = sorted({entity_to_doc[n] for n in nodes if n in entity_to_doc})
    return [d for d in docs if d["id"] in doc_ids]

graph_retrieve("Payment API", hops=1)

[{'id': 'd1', 'text': 'Payment API depends on Auth Service and Redis cache.'},
 {'id': 'd2', 'text': 'Auth Service writes sessions to Redis cache.'},
 {'id': 'd3',
  'text': 'Checkout Service calls Payment API for charge authorization.'},
 {'id': 'd4',
  'text': 'Fraud Engine monitors Checkout Service and Payment API events.'}]

## 4) Hybrid GraphRAG Context Builder

In [5]:
def hybrid_context(query: str, seed_entity: str, k: int = 2, hops: int = 1):
    vec_hits = vector_retrieve(query, k=k)
    graph_hits = graph_retrieve(seed_entity, hops=hops)
    merged = []
    seen = set()
    for doc_id, text, score in vec_hits:
        merged.append({"id": doc_id, "text": text, "source": "vector", "score": score})
        seen.add(doc_id)
    for d in graph_hits:
        if d["id"] not in seen:
            merged.append({"id": d["id"], "text": d["text"], "source": "graph", "score": None})
    return pd.DataFrame(merged)

ctx = hybrid_context("Why can checkout fail when auth degrades?", "Checkout Service", k=2, hops=2)
ctx

,id,text,source,score
0,d2,Auth Service writes sessions to Redis cache.,vector,0.245968
1,d1,Payment API depends on Auth Service and Redis ...,vector,0.236142
2,d3,Checkout Service calls Payment API for charge ...,graph,NaN
3,d4,Fraud Engine monitors Checkout Service and Pay...,graph,NaN


## 5) Grounded Answer Stub

In production this step would call an LLM with structured citations. Here we produce a deterministic explanation from retrieved context.

In [6]:
def grounded_answer(context_df: pd.DataFrame) -> str:
    lines = [f"- ({row.source}) {row.text}" for row in context_df.itertuples()]
    return "Likely checkout risk path includes dependencies across services:\n" + "\n".join(lines)

print(grounded_answer(ctx))

Likely checkout risk path includes dependencies across services:
- (vector) Auth Service writes sessions to Redis cache.
- (vector) Payment API depends on Auth Service and Redis cache.
- (graph) Checkout Service calls Payment API for charge authorization.
- (graph) Fraud Engine monitors Checkout Service and Payment API events.


## Connect to Theory
- Vector retrieval gives semantic recall.
- Graph retrieval gives relational context.
- Hybrid context improves multi-hop question support.

## Case Studies & Exceptions
1. Policy assistant over regulatory docs: graph links between clauses improve citation paths.
2. Fraud analytics: relationship features reveal rings missed by row-wise models.
3. Exception: if corpus is tiny and flat, graph indexing overhead may not pay off.

## Interview Questions & Answers
1. **Q:** What is GraphRAG? **A:** Graph-aware retrieval plus generation grounded in relational context.
2. **Q:** Why hybrid retrieval? **A:** To combine semantic similarity with explicit relationships.
3. **Q:** What is entity resolution risk? **A:** Wrong merges/splits that distort retrieval paths.
4. **Q:** Key GraphRAG metric? **A:** Multi-hop retrieval precision/recall with groundedness.
5. **Q:** Why store provenance? **A:** To support auditability and citation checks.
6. **Q:** When is vector-only enough? **A:** Low-relational, single-hop lookup tasks.
7. **Q:** Main operational challenge? **A:** Graph refresh and schema governance.
8. **Q:** What causes graph noise? **A:** Low-confidence relation extraction.
9. **Q:** Graph DB required? **A:** Not for toy systems, usually yes for scale.
10. **Q:** One-line design rule? **A:** Add graphs when relationships materially affect answer quality.